# 69. The Shortest Path Problem

## Tier 3: The Advanced Algorithm (Metaheuristic Implementation)

### Key assumptions
- Complex shortest path problems with dynamic networks or multiple constraints
- Traditional algorithms may be insufficient for large-scale or multi-objective problems
- Population-based search can explore multiple solution alternatives simultaneously
- Metaheuristic parameters can be tuned for specific problem characteristics

### Approach (step-by-step)
1. **Algorithm Selection**: Implement Moth-Flame Optimization (MFO) inspired by moth navigation
2. **Population Initialization**: Create random moth positions representing potential paths
3. **Fitness Evaluation**: Calculate path quality using distance and constraint penalties
4. **Flame Selection**: Identify best solutions as flames for moth navigation
5. **Spiral Update**: Apply logarithmic spiral movement around flames
6. **Convergence Analysis**: Track population diversity and solution improvement

### What to look for in the results
- Convergence behavior showing cost reduction over iterations
- Population diversity maintenance to avoid local optima
- Comparison with Dijkstra's algorithm for solution quality
- Parameter sensitivity analysis for algorithm tuning

### Concrete example (from the source)
Complex freight network with multiple constraints:
- Expected convergence: Iteration 0 cost = 2850 → Iteration 80 cost = 2150
- Expected optimal path: LA -> Phoenix -> Albuquerque -> Denver -> KC -> Chicago -> NYC
- Expected total distance: 2150 miles with superior performance on complex networks

In [1]:
# Import required libraries for Moth-Flame Optimization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import random
import time
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully
NumPy version: 2.4.3


Libraries imported successfully
NumPy version: 2.4.3


Libraries imported successfully
NumPy version: 2.4.3


Libraries imported successfully
NumPy version: 2.4.3


Libraries imported successfully
NumPy version: 2.4.3


In [ ]:
@dataclass
class ComplexNetwork:
    """Represents a complex freight network with multiple constraints"""
    nodes: Dict[str, Tuple[float, float]]  # node_id -> (x, y) coordinates
    edges: Dict[str, Dict[str, float]]     # adjacency list: node -> {neighbor: distance}
    congestion: Dict[str, Dict[str, float]] # congestion factors for edges
    time_windows: Dict[str, Tuple[int, int]]  # node -> (earliest, latest) arrival
    node_names: Dict[str, str]             # node_id -> full name

@dataclass
class Moth:
    """Represents a moth (potential solution) in MFO algorithm"""
    position: List[int]     # sequence of node indices representing a path
    fitness: float         # fitness value (inverse of total cost)
    valid: bool           # whether the path is valid

@dataclass
class MFOResult:
    """Results from Moth-Flame Optimization"""
    best_path: List[str]
    best_cost: float
    convergence_history: List[float]
    diversity_history: List[float]
    computation_time: float
    iterations: int

print("Data structures defined")

In [ ]:
def create_complex_freight_network() -> ComplexNetwork:
    """Create a complex freight network with multiple constraints"""
    
    # Extended network with more nodes and complexity
    nodes = {
        'LA': (0, 0),        # Los Angeles
        'PHX': (2, 1),       # Phoenix
        'ALB': (3, 2),       # Albuquerque
        'DEN': (1, 2),       # Denver
        'SALT': (2, 3),      # Salt Lake City
        'KCM': (4, 2),       # Kansas City
        'DAL': (3, 1),       # Dallas
        'OKC': (4, 1.5),     # Oklahoma City
        'CHI': (5, 2),       # Chicago
        'DET': (5.5, 2.5),   # Detroit
        'MIN': (4.5, 3),     # Minneapolis
        'STL': (5, 1.5),     # St. Louis
        'IND': (6, 2),       # Indianapolis
        'COL': (6.5, 2.5),   # Columbus
        'PIT': (7, 2),       # Pittsburgh
        'ATL': (6, 1),       # Atlanta
        'NYC': (8, 2),       # New York
        'BOS': (9, 3),       # Boston
        'WAS': (7.5, 1.5)    # Washington DC
    }
    
    node_names = {
        'LA': 'Los Angeles', 'PHX': 'Phoenix', 'ALB': 'Albuquerque', 'DEN': 'Denver',
        'SALT': 'Salt Lake City', 'KCM': 'Kansas City', 'DAL': 'Dallas', 'OKC': 'Oklahoma City',
        'CHI': 'Chicago', 'DET': 'Detroit', 'MIN': 'Minneapolis', 'STL': 'St. Louis',
        'IND': 'Indianapolis', 'COL': 'Columbus', 'PIT': 'Pittsburgh', 'ATL': 'Atlanta',
        'NYC': 'New York', 'BOS': 'Boston', 'WAS': 'Washington DC'
    }
    
    # Base distances
    edges = {
        'LA': {'PHX': 370, 'DEN': 1015},
        'PHX': {'LA': 370, 'ALB': 270, 'DEN': 586, 'KCM': 1150},
        'ALB': {'PHX': 270, 'DEN': 230, 'KCM': 560, 'SALT': 520},
        'DEN': {'LA': 1015, 'PHX': 586, 'ALB': 230, 'SALT': 530, 'KCM': 600, 'MIN': 920},
        'SALT': {'DEN': 530, 'ALB': 520, 'MIN': 1300, 'KCM': 980},
        'KCM': {'PHX': 1150, 'ALB': 560, 'DEN': 600, 'SALT': 980, 'OKC': 350, 'STL': 250, 'CHI': 515, 'MIN': 398},
        'DAL': {'OKC': 200, 'STL': 450, 'ATL': 795},
        'OKC': {'DAL': 200, 'KCM': 350, 'STL': 500},
        'CHI': {'KCM': 515, 'MIN': 409, 'DET': 281, 'STL': 300, 'IND': 180},
        'DET': {'CHI': 281, 'MIN': 527, 'COL': 200},
        'MIN': {'DEN': 920, 'KCM': 398, 'SALT': 1300, 'CHI': 409, 'DET': 527},
        'STL': {'KCM': 250, 'OKC': 500, 'DAL': 450, 'CHI': 300, 'IND': 250},
        'IND': {'CHI': 180, 'STL': 250, 'COL': 180, 'ATL': 430},
        'COL': {'DET': 200, 'IND': 180, 'PIT': 180},
        'PIT': {'COL': 180, 'ATL': 680, 'NYC': 370, 'WAS': 250},
        'ATL': {'IND': 430, 'PIT': 680, 'WAS': 640, 'NYC': 888},
        'NYC': {'PIT': 370, 'WAS': 230, 'BOS': 215},
        'BOS': {'NYC': 215},
        'WAS': {'PIT': 250, 'ATL': 640, 'NYC': 230}
    }
    
    # Congestion factors (1.0 = no congestion, higher = more congestion)
    congestion = {
        'LA': {'PHX': 1.2, 'DEN': 1.0},
        'PHX': {'LA': 1.2, 'ALB': 1.1, 'DEN': 1.0, 'KCM': 1.3},
        'ALB': {'PHX': 1.1, 'DEN': 1.0, 'KCM': 1.2, 'SALT': 1.1},
        'DEN': {'LA': 1.0, 'PHX': 1.0, 'ALB': 1.0, 'SALT': 1.1, 'KCM': 1.1, 'MIN': 1.2},
        'SALT': {'DEN': 1.1, 'ALB': 1.1, 'MIN': 1.3, 'KCM': 1.2},
        'KCM': {'PHX': 1.3, 'ALB': 1.2, 'DEN': 1.1, 'SALT': 1.2, 'OKC': 1.0, 'STL': 1.1, 'CHI': 1.2, 'MIN': 1.1},
        'DAL': {'OKC': 1.1, 'STL': 1.2, 'ATL': 1.3},
        'OKC': {'DAL': 1.1, 'KCM': 1.0, 'STL': 1.1},
        'CHI': {'KCM': 1.2, 'MIN': 1.1, 'DET': 1.0, 'STL': 1.1, 'IND': 1.0},
        'DET': {'CHI': 1.0, 'MIN': 1.1, 'COL': 1.0},
        'MIN': {'DEN': 1.2, 'KCM': 1.1, 'SALT': 1.3, 'CHI': 1.1, 'DET': 1.1},
        'STL': {'KCM': 1.1, 'OKC': 1.1, 'DAL': 1.2, 'CHI': 1.1, 'IND': 1.0},
        'IND': {'CHI': 1.0, 'STL': 1.0, 'COL': 1.0, 'ATL': 1.1},
        'COL': {'DET': 1.0, 'IND': 1.0, 'PIT': 1.0},
        'PIT': {'COL': 1.0, 'ATL': 1.2, 'NYC': 1.3, 'WAS': 1.1},
        'ATL': {'IND': 1.1, 'PIT': 1.2, 'WAS': 1.2, 'NYC': 1.4},
        'NYC': {'PIT': 1.3, 'WAS': 1.1, 'BOS': 1.0},
        'BOS': {'NYC': 1.0},
        'WAS': {'PIT': 1.1, 'ATL': 1.2, 'NYC': 1.1}
    }
    
    # Time windows (operating hours in 24-hour format)
    time_windows = {
        'LA': (0, 24), 'PHX': (0, 24), 'ALB': (6, 22), 'DEN': (0, 24),
        'SALT': (0, 24), 'KCM': (0, 24), 'DAL': (0, 24), 'OKC': (0, 24),
        'CHI': (0, 24), 'DET': (6, 22), 'MIN': (0, 24), 'STL': (0, 24),
        'IND': (0, 24), 'COL': (6, 22), 'PIT': (0, 24), 'ATL': (0, 24),
        'NYC': (0, 24), 'BOS': (6, 22), 'WAS': (0, 24)
    }
    
    return ComplexNetwork(nodes, edges, congestion, time_windows, node_names)

# Create the complex network
complex_network = create_complex_freight_network()
print(f"Complex network created with {len(complex_network.nodes)} nodes")
print(f"Network includes congestion factors and time windows")
print(f"Total edges: {sum(len(neighbors) for neighbors in complex_network.edges.values())}")

In [ ]:
def calculate_path_cost(network: ComplexNetwork, path: List[str]) -> float:
    """Calculate total cost of a path including congestion penalties"""
    
    if len(path) < 2:
        return float('inf')
    
    total_cost = 0
    
    for i in range(len(path) - 1):
        from_node = path[i]
        to_node = path[i + 1]
        
        # Check if edge exists
        if from_node not in network.edges or to_node not in network.edges[from_node]:
            return float('inf')
        
        # Base distance
        base_distance = network.edges[from_node][to_node]
        
        # Apply congestion factor
        congestion_factor = network.congestion.get(from_node, {}).get(to_node, 1.0)
        
        # Time window penalty (higher penalty for arriving outside preferred hours)
        time_penalty = 1.0
        if to_node in network.time_windows:
            earliest, latest = network.time_windows[to_node]
            # Simplified time calculation (assume 60 mph average speed)
            cumulative_time = sum(network.edges[path[j]][path[j+1]] / 60 for j in range(i + 1))
            arrival_hour = (cumulative_time % 24)
            
            if arrival_hour < earliest or arrival_hour > latest:
                time_penalty = 1.5  # 50% penalty for off-hours arrival
        
        segment_cost = base_distance * congestion_factor * time_penalty
        total_cost += segment_cost
    
    return total_cost

def is_valid_path(network: ComplexNetwork, path: List[str]) -> bool:
    """Check if a path is valid (connected sequence)"""
    
    if len(path) < 2:
        return False
    
    for i in range(len(path) - 1):
        from_node = path[i]
        to_node = path[i + 1]
        
        if from_node not in network.edges or to_node not in network.edges[from_node]:
            return False
    
    return True

def generate_random_path(network: ComplexNetwork, source: str, target: str) -> List[str]:
    """Generate a random valid path from source to target"""
    
    path = [source]
    current = source
    visited = {source}
    
    while current != target:
        # Get available neighbors
        neighbors = [n for n in network.edges.get(current, {}) if n not in visited]
        
        if not neighbors:
            # Backtrack if stuck
            if len(path) > 1:
                path.pop()
                current = path[-1]
                continue
            else:
                # Restart with different approach
                return generate_greedy_path(network, source, target)
        
        # Choose next node (weighted by inverse distance for better paths)
        weights = [1.0 / network.edges[current][neighbor] for neighbor in neighbors]
        total_weight = sum(weights)
        probabilities = [w / total_weight for w in weights]
        
        next_node = random.choices(neighbors, weights=probabilities)[0]
        
        path.append(next_node)
        visited.add(next_node)
        current = next_node
        
        # Prevent infinite loops
        if len(path) > len(network.nodes):
            return generate_greedy_path(network, source, target)
    
    return path

def generate_greedy_path(network: ComplexNetwork, source: str, target: str) -> List[str]:
    """Generate a greedy path (always choose closest neighbor towards target)"""
    
    path = [source]
    current = source
    visited = {source}
    
    while current != target:
        neighbors = [n for n in network.edges.get(current, {}) if n not in visited]
        
        if not neighbors:
            break
        
        # Choose neighbor closest to target
        best_neighbor = None
        best_distance = float('inf')
        
        for neighbor in neighbors:
            # Simple heuristic: straight-line distance to target
            target_coords = network.nodes[target]
            neighbor_coords = network.nodes[neighbor]
            distance = np.sqrt((target_coords[0] - neighbor_coords[0])**2 + 
                             (target_coords[1] - neighbor_coords[1])**2)
            
            if distance < best_distance:
                best_distance = distance
                best_neighbor = neighbor
        
        if best_neighbor:
            path.append(best_neighbor)
            visited.add(best_neighbor)
            current = best_neighbor
        else:
            break
    
    return path

print("Path utility functions defined")

In [ ]:
def initialize_moth_population(network: ComplexNetwork, source: str, target: str, 
                             population_size: int) -> List[Moth]:
    """Initialize moth population with diverse paths"""
    
    moths = []
    
    # Generate initial population with mixed strategies
    for i in range(population_size):
        if i < population_size // 3:
            # Random paths for diversity
            path = generate_random_path(network, source, target)
        elif i < 2 * population_size // 3:
            # Greedy paths for quality
            path = generate_greedy_path(network, source, target)
        else:
            # Semi-random paths (greedy with some randomness)
            path = generate_greedy_path(network, source, target)
            if len(path) > 3:
                # Introduce some randomness in the middle
                mid_point = len(path) // 2
                if mid_point > 0 and mid_point < len(path) - 1:
                    current = path[mid_point]
                    neighbors = [n for n in network.edges.get(current, {}) 
                               if n not in path[:mid_point+1] and n not in path[mid_point+1:]]
                    if neighbors:
                        new_middle = random.choice(neighbors)
                        path[mid_point] = new_middle
        
        # Calculate fitness (inverse of cost, higher is better)
        if is_valid_path(network, path):
            cost = calculate_path_cost(network, path)
            fitness = 1.0 / (1.0 + cost)  # Avoid division by zero
        else:
            fitness = 0.0
        
        moth = Moth(path, fitness, is_valid_path(network, path))
        moths.append(moth)
    
    return moths

def logarithmic_spiral_update(moth_position: List[str], flame_position: List[str], 
                            b: float = 1.0, t: float = None) -> List[str]:
    """Update moth position using logarithmic spiral around flame"""
    
    if t is None:
        t = random.uniform(-1, 1)
    
    # For path representation, we'll use a simplified spiral update
    # that balances between current position and flame position
    
    new_position = []
    
    # Determine the shorter path length
    max_length = min(len(moth_position), len(flame_position))
    
    for i in range(max_length):
        # Spiral update formula adapted for discrete paths
        distance_ratio = i / max(max_length - 1, 1)
        spiral_factor = np.exp(b * t) * np.cos(2 * np.pi * t)
        
        # Blend between moth and flame based on spiral factor
        if random.random() < abs(spiral_factor):
            if i < len(flame_position):
                new_position.append(flame_position[i])
            else:
                new_position.append(moth_position[i])
        else:
            new_position.append(moth_position[i])
    
    # Ensure we reach the target
    if new_position and flame_position:
        if new_position[-1] != flame_position[-1]:
            new_position.append(flame_position[-1])
    
    return new_position

def calculate_population_diversity(moths: List[Moth]) -> float:
    """Calculate population diversity based on path differences"""
    
    if len(moths) < 2:
        return 0.0
    
    total_differences = 0
    comparisons = 0
    
    for i in range(len(moths)):
        for j in range(i + 1, len(moths)):
            if moths[i].valid and moths[j].valid:
                # Calculate path difference (simplified)
                path1_set = set(moths[i].position)
                path2_set = set(moths[j].position)
                
                if path1_set and path2_set:
                    intersection = len(path1_set.intersection(path2_set))
                    union = len(path1_set.union(path2_set))
                    similarity = intersection / union if union > 0 else 0
                    difference = 1.0 - similarity
                    total_differences += difference
                    comparisons += 1
    
    return total_differences / comparisons if comparisons > 0 else 0.0

print("MFO algorithm components defined")

In [ ]:
def moth_flame_optimization(network: ComplexNetwork, source: str, target: str,
                          population_size: int = 30, max_iterations: int = 100,
                          spiral_constant: float = 1.0) -> MFOResult:
    """Implement Moth-Flame Optimization algorithm for shortest path"""
    
    start_time = time.time()
    
    # Initialize moth population
    moths = initialize_moth_population(network, source, target, population_size)
    
    convergence_history = []
    diversity_history = []
    
    print(f"Starting MFO with {population_size} moths, {max_iterations} iterations")
    
    for iteration in range(max_iterations):
        # Sort moths by fitness (descending)
        moths.sort(key=lambda m: m.fitness, reverse=True)
        
        # Update convergence history
        best_cost = 1.0 / moths[0].fitness - 1.0 if moths[0].fitness > 0 else float('inf')
        convergence_history.append(best_cost)
        
        # Calculate population diversity
        diversity = calculate_population_diversity(moths)
        diversity_history.append(diversity)
        
        # Print progress every 20 iterations
        if iteration % 20 == 0:
            print(f"Iteration {iteration}: Best cost = {best_cost:.1f}, Diversity = {diversity:.3f}")
        
        # Number of flames (decreases over time)
        num_flames = max(1, int(population_size * (1 - iteration / max_iterations)))
        
        # Select flames (best moths)
        flames = moths[:num_flames]
        
        # Update moth positions
        new_moths = []
        
        for i, moth in enumerate(moths):
            if i < num_flames:
                # Best moths stay as flames
                new_moths.append(moth)
            else:
                # Other moths spiral around flames
                flame_index = i % num_flames
                flame_position = flames[flame_index].position
                
                # Apply spiral update
                new_position = logarithmic_spiral_update(
                    moth.position, flame_position, spiral_constant
                )
                
                # Ensure valid path
                if not is_valid_path(network, new_position):
                    # Fallback to greedy path
                    new_position = generate_greedy_path(network, source, target)
                
                # Calculate new fitness
                if is_valid_path(network, new_position):
                    cost = calculate_path_cost(network, new_position)
                    fitness = 1.0 / (1.0 + cost)
                else:
                    fitness = 0.0
                
                new_moth = Moth(new_position, fitness, is_valid_path(network, new_position))
                new_moths.append(new_moth)
        
        moths = new_moths
        
        # Apply local search to best solution occasionally
        if iteration % 10 == 0 and moths:
            best_moth = max(moths, key=lambda m: m.fitness)
            if best_moth.valid:
                # Try to improve best solution with local search
                improved_path = local_search_improvement(network, best_moth.position)
                if is_valid_path(network, improved_path):
                    new_cost = calculate_path_cost(network, improved_path)
                    old_cost = 1.0 / best_moth.fitness - 1.0 if best_moth.fitness > 0 else float('inf')
                    
                    if new_cost < old_cost:
                        new_fitness = 1.0 / (1.0 + new_cost)
                        best_moth.position = improved_path
                        best_moth.fitness = new_fitness
    
    # Get final best solution
    best_moth = max(moths, key=lambda m: m.fitness)
    best_cost = 1.0 / best_moth.fitness - 1.0 if best_moth.fitness > 0 else float('inf')
    
    computation_time = time.time() - start_time
    
    return MFOResult(
        best_path=best_moth.position,
        best_cost=best_cost,
        convergence_history=convergence_history,
        diversity_history=diversity_history,
        computation_time=computation_time,
        iterations=max_iterations
    )

def local_search_improvement(network: ComplexNetwork, path: List[str]) -> List[str]:
    """Apply local search to improve a path"""
    
    if len(path) < 3:
        return path
    
    best_path = path.copy()
    best_cost = calculate_path_cost(network, best_path)
    
    # Try 2-opt moves (swap two intermediate nodes)
    for i in range(1, len(path) - 2):
        for j in range(i + 1, len(path) - 1):
            new_path = path[:i] + path[i:j+1][::-1] + path[j+1:]
            
            if is_valid_path(network, new_path):
                new_cost = calculate_path_cost(network, new_path)
                if new_cost < best_cost:
                    best_path = new_path
                    best_cost = new_cost
    
    return best_path

print("MFO algorithm defined")

In [ ]:
# Run Moth-Flame Optimization
source = 'LA'
target = 'NYC'

mfo_result = moth_flame_optimization(
    complex_network, source, target,
    population_size=30,
    max_iterations=100,
    spiral_constant=1.0
)

print("\n=== MOTH-FLAME OPTIMIZATION RESULTS ===")
print(f"Source: {source} ({complex_network.node_names[source]})")
print(f"Target: {target} ({complex_network.node_names[target]})")
print(f"Best path: {' -> '.join(mfo_result.best_path)}")
print(f"Best cost: {mfo_result.best_cost:.1f}")
print(f"Computation time: {mfo_result.computation_time:.2f} seconds")
print(f"Iterations: {mfo_result.iterations}")

print("\nPath details with congestion and time factors:")
for i in range(len(mfo_result.best_path) - 1):
    from_node = mfo_result.best_path[i]
    to_node = mfo_result.best_path[i + 1]
    
    base_distance = complex_network.edges[from_node][to_node]
    congestion = complex_network.congestion[from_node][to_node]
    effective_distance = base_distance * congestion
    
    print(f"  {from_node} -> {to_node}: {base_distance:.0f}mi × {congestion:.1f} = {effective_distance:.1f}mi")

print(f"\nTotal effective distance: {mfo_result.best_cost:.1f} miles")

In [ ]:
def visualize_mfo_results(network: ComplexNetwork, result: MFOResult, source: str, target: str):
    """Visualize MFO algorithm results and convergence"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Network with MFO solution
    ax1.set_title('MFO Optimal Path on Complex Network', fontsize=14, fontweight='bold')
    
    # Draw all edges with congestion coloring
    for from_node, neighbors in network.edges.items():
        for to_node, distance in neighbors.items():
            from_coords = network.nodes[from_node]
            to_coords = network.nodes[to_node]
            congestion = network.congestion[from_node][to_node]
            
            # Color based on congestion level
            if congestion <= 1.0:
                color = 'green'
            elif congestion <= 1.2:
                color = 'orange'
            else:
                color = 'red'
            
            # Check if edge is in optimal path
            is_optimal = (from_node in result.best_path and to_node in result.best_path and 
                         result.best_path.index(from_node) + 1 == result.best_path.index(to_node))
            
            linewidth = 4 if is_optimal else 1
            alpha = 0.9 if is_optimal else 0.3
            
            ax1.annotate('', xy=to_coords, xytext=from_coords,
                       arrowprops=dict(arrowstyle='->', color=color, lw=linewidth, alpha=alpha))
    
    # Draw nodes
    for node_id, coords in network.nodes.items():
        color = 'red' if node_id == source else 'blue' if node_id == target else 'lightgreen'
        size = 400 if node_id in result.best_path else 300
        ax1.scatter(coords[0], coords[1], s=size, c=color, edgecolors='black', linewidth=2, zorder=5)
        ax1.text(coords[0], coords[1]-0.2, node_id, fontsize=8, ha='center', fontweight='bold')
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal')
    
    # Plot 2: Convergence history
    ax2.set_title('MFO Convergence History', fontsize=14, fontweight='bold')
    iterations = list(range(len(result.convergence_history)))
    ax2.plot(iterations, result.convergence_history, 'b-', linewidth=2, label='Best Cost')
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Path Cost')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    # Add convergence milestones
    milestones = [0, 20, 40, 60, 80, 100]
    for milestone in milestones:
        if milestone < len(result.convergence_history):
            cost = result.convergence_history[milestone]
            ax2.annotate(f'{cost:.0f}', 
                        (milestone, cost), 
                        textcoords="offset points", 
                        xytext=(0,10), 
                        ha='center', 
                        fontsize=8)
    
    # Plot 3: Population diversity
    ax3.set_title('Population Diversity Over Time', fontsize=14, fontweight='bold')
    ax3.plot(iterations, result.diversity_history, 'g-', linewidth=2, label='Diversity')
    ax3.set_xlabel('Iteration')
    ax3.set_ylabel('Population Diversity')
    ax3.set_ylim(0, 1)
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    
    # Add diversity threshold line
    ax3.axhline(y=0.1, color='red', linestyle='--', alpha=0.7, label='Low Diversity Threshold')
    ax3.legend()
    
    # Plot 4: Algorithm performance comparison
    ax4.set_title('Algorithm Performance Metrics', fontsize=14, fontweight='bold')
    
    # Compare with expected results from source
    expected_costs = [2850, 2400, 2200, 2150, 2150]
    expected_iterations = [0, 20, 40, 60, 80]
    
    # Sample our convergence at same points
    our_costs = []
    for it in expected_iterations:
        if it < len(result.convergence_history):
            our_costs.append(result.convergence_history[it])
        else:
            our_costs.append(result.convergence_history[-1])
    
    x = np.arange(len(expected_iterations))
    width = 0.35
    
    bars1 = ax4.bar(x - width/2, expected_costs, width, label='Expected (Source)', alpha=0.8, color='lightblue')
    bars2 = ax4.bar(x + width/2, our_costs, width, label='MFO Result', alpha=0.8, color='lightcoral')
    
    ax4.set_xlabel('Iteration Milestone')
    ax4.set_ylabel('Path Cost')
    ax4.set_xticks(x)
    ax4.set_xticklabels([f'It {it}' for it in expected_iterations])
    ax4.legend()
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height + 20,
                    f'{height:.0f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()

# Visualize MFO results
visualize_mfo_results(complex_network, mfo_result, source, target)

In [ ]:
def parameter_sensitivity_analysis(network: ComplexNetwork, source: str, target: str):
    """Analyze sensitivity of MFO parameters"""
    
    print("=== PARAMETER SENSITIVITY ANALYSIS ===")
    
    # Test different population sizes
    population_sizes = [10, 20, 30, 40, 50]
    population_results = []
    
    print("\nTesting different population sizes:")
    for pop_size in population_sizes:
        result = moth_flame_optimization(
            network, source, target,
            population_size=pop_size,
            max_iterations=50,  # Reduced for faster testing
            spiral_constant=1.0
        )
        
        population_results.append({
            'population_size': pop_size,
            'best_cost': result.best_cost,
            'computation_time': result.computation_time,
            'final_diversity': result.diversity_history[-1] if result.diversity_history else 0
        })
        
        print(f"  Population {pop_size:2d}: Cost = {result.best_cost:6.1f}, "
              f"Time = {result.computation_time:5.2f}s, Diversity = {result.diversity_history[-1]:.3f}")
    
    # Test different spiral constants
    spiral_constants = [0.5, 1.0, 1.5, 2.0]
    spiral_results = []
    
    print("\nTesting different spiral constants:")
    for spiral_const in spiral_constants:
        result = moth_flame_optimization(
            network, source, target,
            population_size=30,
            max_iterations=50,
            spiral_constant=spiral_const
        )
        
        spiral_results.append({
            'spiral_constant': spiral_const,
            'best_cost': result.best_cost,
            'computation_time': result.computation_time,
            'convergence_rate': (result.convergence_history[0] - result.convergence_history[-1]) / len(result.convergence_history) if len(result.convergence_history) > 1 else 0
        })
        
        print(f"  Spiral {spiral_const:3.1f}: Cost = {result.best_cost:6.1f}, "
              f"Time = {result.computation_time:5.2f}s, Rate = {spiral_results[-1]['convergence_rate']:.1f}")
    
    return population_results, spiral_results

# Run parameter sensitivity analysis
pop_results, spiral_results = parameter_sensitivity_analysis(complex_network, source, target)

In [ ]:
def compare_with_dijkstra_and_expected():
    """Compare MFO results with Dijkstra and expected results"""
    
    print("=== COMPREHENSIVE COMPARISON ===")
    
    # Run Dijkstra for comparison (simplified network without congestion)
    simple_network = {
        node: {neighbor: distance for neighbor, distance in neighbors.items()}
        for node, neighbors in complex_network.edges.items()
    }
    
    # Use a simple Dijkstra implementation for comparison
    def simple_dijkstra(edges, source, target):
        distances = {node: float('inf') for node in edges}
        distances[source] = 0
        predecessors = {node: None for node in edges}
        unvisited = set(edges.keys())
        
        while unvisited:
            current = min(unvisited, key=lambda x: distances[x])
            unvisited.remove(current)
            
            if current == target:
                break
            
            for neighbor, distance in edges[current].items():
                if neighbor in unvisited:
                    new_distance = distances[current] + distance
                    if new_distance < distances[neighbor]:
                        distances[neighbor] = new_distance
                        predecessors[neighbor] = current
        
        # Reconstruct path
        path = []
        current = target
        while current is not None:
            path.append(current)
            current = predecessors[current]
        
        return path[::-1], distances[target]
    
    dijkstra_path, dijkstra_cost = simple_dijkstra(simple_network, source, target)
    
    print("Comparison Results:")
    print(f"\nDijkstra (no congestion):")
    print(f"  Path: {' -> '.join(dijkstra_path)}")
    print(f"  Cost: {dijkstra_cost:.1f}")
    
    print(f"\nMFO (with congestion):")
    print(f"  Path: {' -> '.join(mfo_result.best_path)}")
    print(f"  Cost: {mfo_result.best_cost:.1f}")
    
    print(f"\nExpected (from source document):")
    print(f"  Path: LA -> Phoenix -> Albuquerque -> Denver -> KC -> Chicago -> NYC")
    print(f"  Cost: 2150")
    
    # Analysis
    print("\n=== ANALYSIS ===")
    
    # Path similarity analysis
    expected_path = ['LA', 'PHX', 'ALB', 'DEN', 'KCM', 'CHI', 'NYC']
    
    def path_similarity(path1, path2):
        set1, set2 = set(path1), set(path2)
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))
        return intersection / union if union > 0 else 0
    
    mfo_expected_similarity = path_similarity(mfo_result.best_path, expected_path)
    dijkstra_expected_similarity = path_similarity(dijkstra_path, expected_path)
    
    print(f"Path similarity to expected:")
    print(f"  MFO: {mfo_expected_similarity:.2f}")
    print(f"  Dijkstra: {dijkstra_expected_similarity:.2f}")
    
    # Performance comparison
    print(f"\nPerformance characteristics:")
    print(f"  MFO handles congestion: ✓")
    print(f"  Dijkstra handles congestion: ✗")
    print(f"  MFO computation time: {mfo_result.computation_time:.2f}s")
    print(f"  MFO explores multiple alternatives: ✓")
    print(f"  Dijkstra finds single optimal: ✓")
    
    # When to use each approach
    print(f"\n=== RECOMMENDATIONS ===")
    print(f"Use Dijkstra when:")
    print(f"  • Network has static, non-negative weights")
    print(f"  • Single optimal solution is required")
    print(f"  • Computational efficiency is critical")
    print(f"  • Problem is well-defined and unconstrained")
    
    print(f"\nUse MFO when:")
    print(f"  • Network has dynamic factors (congestion, time windows)")
    print(f"  • Multiple good solutions are valuable")
    print(f"  • Problem has complex constraints")
    print(f"  • Exploration of solution space is important")
    print(f"  • Traditional algorithms are insufficient")

# Run comprehensive comparison
compare_with_dijkstra_and_expected()

### Why this Tier exists vs earlier Tiers
Tier 3 provides advanced capabilities beyond mathematical formulation and classic heuristics:
- **Complex Constraint Handling**: Incorporates congestion factors and time windows naturally
- **Population-Based Search**: Explores multiple solution alternatives simultaneously
- **Adaptive Behavior**: Spiral update mechanism balances exploration and exploitation
- **Robustness**: Handles dynamic network conditions and multi-objective optimization
- **Solution Diversity**: Maintains population diversity to avoid local optima

### Pros / Cons vs earlier Tiers
**Pros:**
- Handles complex constraints (congestion, time windows) naturally
- Explores multiple high-quality solutions simultaneously
- Adapts to dynamic network conditions
- Robust against local optima through population diversity
- Suitable for multi-objective optimization
- Provides solution alternatives for contingency planning

**Cons:**
- Higher computational cost than Dijkstra's algorithm
- No guarantee of finding the global optimum
- Requires parameter tuning for best performance
- More complex implementation and understanding
- Convergence can be slow for large problems
- Results may vary between runs due to stochastic nature

### When to use this Tier
- **Dynamic networks** with time-varying edge weights
- **Multi-constraint problems** (congestion, time windows, capacity)
- **Multi-objective optimization** requiring trade-off analysis
- **Contingency planning** where multiple good solutions are valuable
- **Complex real-world routing** with many practical constraints
- **When traditional algorithms** are insufficient or inflexible

### Key takeaways
Moth-Flame Optimization demonstrates advanced metaheuristic capabilities:
- Nature-inspired spiral navigation provides effective search mechanism
- Population diversity maintains exploration capabilities
- Adaptive flame selection balances convergence and diversity
- Complex constraints can be incorporated naturally in fitness evaluation
- Provides practical solutions for real-world complex routing problems